In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
gdb_file_path = r"/Users/xiaoliu/Work/Project/I-GUIDE/IGUIDE_Aging_Dam-selected/NTAD_North_American_Roads/North_American_Roads.shp"
layer_name = "North_American_Roads"  # Replace with the actual layer name

# Read the specific layer into a GeoDataFrame
try:
    gdf = gpd.read_file(gdb_file_path, layer=layer_name)
    print(f"Successfully loaded layer: {layer_name}")
except Exception as e:
    print(f"Error loading GDB layer: {e}")

Successfully loaded layer: North_American_Roads


In [3]:
gdf.columns

Index(['ID', 'DIR', 'LENGTH', 'LINKID', 'COUNTRY', 'JURISCODE', 'JURISNAME',
       'ROADNUM', 'ROADNAME', 'ADMIN', 'SURFACE', 'LANES', 'SPEEDLIM', 'CLASS',
       'NHS', 'BORDER', 'geometry'],
      dtype='object')

In [4]:
columns_rename = ['ID', 'DIR', 'LENGTH', 'LINKID', 'COUNTRY', 'JURISCODE', 'JURISNAME',
       'ROADNUM', 'ROADNAME', 'ADMIN', 'SURFACE', 'LANES', 'SPEEDLIM', 'CLASS',
       'NHS', 'BORDER', 'geom']

In [5]:
gdf.columns = columns_rename

In [6]:
columns_to_keep = ['ID', 'COUNTRY', 'JURISCODE', 'JURISNAME',
       'ROADNUM', 'ROADNAME', 'ADMIN', 'SURFACE', 'CLASS', 'geom']

In [7]:
gdf = gdf[columns_to_keep]

In [8]:
gdf = gdf.set_geometry("geom")

In [9]:
target_crs = "EPSG:4326"
reprojected_gdf = gdf.to_crs(target_crs)

print(f"Original CRS: {gdf.crs}")
print(f"New CRS: {reprojected_gdf.crs}")

Original CRS: EPSG:4326
New CRS: EPSG:4326


In [10]:
columns_to_group_by = ['ROADNUM', 'JURISCODE']

In [11]:
print(len(reprojected_gdf))
reprojected_gdf = reprojected_gdf[reprojected_gdf.geometry.notna()]
reprojected_gdf = reprojected_gdf[~reprojected_gdf.geometry.is_empty]
print(len(reprojected_gdf))

720055
720054


In [12]:
reprojected_gdf['geom'] = reprojected_gdf.geometry.make_valid()

In [13]:
dissolved_gdf = reprojected_gdf.dissolve(by=columns_to_group_by)

/opt/homebrew/Caskroom/miniforge/base/envs/postgis-env/lib/python3.13/site-packages/shapely/set_operations.py:553: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


In [14]:
len(dissolved_gdf)

12358

In [26]:
dissolved_gdf = dissolved_gdf.reset_index()
dissolved_gdf.columns

Index(['ROADNUM', 'JURISCODE', 'geom', 'ID', 'COUNTRY', 'JURISNAME',
       'ROADNAME', 'ADMIN', 'SURFACE', 'CLASS'],
      dtype='object')

In [27]:
dissolved_gdf.head()

,ROADNUM,JURISCODE,geom,ID,COUNTRY,JURISNAME,ROADNAME,ADMIN,SURFACE,CLASS
0,0,03_22,"MULTILINESTRING ((-100.27065 20.39909, -100.27...",1122046,3,Quer2taro,Huimilpan - Escolasticas,State,Paved,3
1,001,03_26,"LINESTRING (-114.8261 32.47003, -114.8261 32.4...",1107203,3,Sonora,ej san luis - ec [slrc - est riito - golf sta ...,State,Paved,2
2,002,03_26,"MULTILINESTRING ((-109.58149 31.3134, -109.581...",1108071,3,Sonora,agua prieta - cananea,Federal,Paved,1
3,003,03_26,"MULTILINESTRING ((-112.20476 29.52905, -112.20...",1107888,3,Sonora,calle 36 norte,State,Paved,2
4,005,03_26,"MULTILINESTRING ((-114.9269 32.24435, -114.926...",1107287,3,Sonora,lagunitas independencias,State,Paved,3


In [28]:
import psycopg2


# ----------------------------
# Connect to PostGIS
# ----------------------------
conn = psycopg2.connect(
    dbname="utahdaminundationprofiles_aug9_2025",
    user="admin",
    password="admin",
    host="localhost",
    port=5432
)

In [29]:
from sqlalchemy import create_engine
import geopandas as gpd

# --- Assume these variables are derived from your psycopg2 connection details ---
# You must provide these credentials instead of the psycopg2 connection object itself
DB_USER = "admin"
DB_PASS = "admin"
DB_HOST = "localhost"
DB_PORT = "5432"  # Standard PostgreSQL port
DB_NAME = "utahdaminundationprofiles_aug9_2025"
table_name = "transportation"

# Assuming 'merged_gdf' is your GeoDataFrame

# 1. CONSTRUCT THE POSTGRESQL CONNECTION URL
db_url = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# 2. CREATE THE SQLALCHEMY ENGINE
# GeoPandas requires a SQLAlchemy Engine object for to_postgis()
engine = create_engine(db_url)

# 3. UPLOAD THE GEODATAFRAME
try:
    dissolved_gdf.to_postgis(
        name=table_name,
        con=engine,          # Use the SQLAlchemy engine
        if_exists='replace', # or 'append'
        index=False 
    )
    print(f"Successfully uploaded GeoDataFrame to PostGIS table: {table_name}")
except Exception as e:
    print(f"Error uploading to PostGIS: {e}")

Successfully uploaded GeoDataFrame to PostGIS table: transportation


In [30]:
conn.close()